## **Customer Shopping Behavior**

#### **Import Libraries**

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'customer_shopping_behavior.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project data folder.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / 'data' / 'customer_shopping_behavior.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset path: {DATA_PATH}')


#### **Import Raw Data**

In [ ]:
df = pd.read_csv(DATA_PATH)

df.head()

#### **Sample Data**

In [ ]:
df.info()

#### **Size of Data**

In [ ]:
print("Size of Data:", df.shape)

#### **Field Information**

In [ ]:
df.describe(include='all')

#### **Searching for null data**

In [ ]:
df.isnull().sum()

#### **Using median of the Review Ratings per category to fill the null data**

In [ ]:
df["Review Rating"] = df.groupby("Category")['Review Rating'].transform(lambda x: x.fillna(x.median()))

#### **Don't have more null values**

In [ ]:
df.isnull().sum()

#### **Standardizing the columns**

In [ ]:
# Standardize column names using lowercase snake_case.
df.columns = (
    df.columns
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

df = df.rename(columns={'purchase_amount_usd': 'purchase_amount'})
df.columns

In [ ]:
df.columns

#### **Grouping the ages in four labels, and creating a new column for that**

In [ ]:
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels).astype(str)

df[['age', 'age_group']].head(10)

In [ ]:
df[['age', 'age_group']].head(10)

#### **Creating column purchase frequency days**

In [ ]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly' : 7,
    'Monthly' : 30,
    'Quarterly' : 90,
    'Bi-Weekly' : 14,
    'Annually' : 365,
    'Every 3 Months' : 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

#### **The column discount applied is equal of the promo_code_used, so remove one of then**

In [ ]:
df[['discount_applied', 'promo_code_used']].head(10)

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
df = df.drop('promo_code_used', axis = 1)

In [ ]:
df.columns

#### **Importing data into the Postgree Database**

**PostgreSQL setup:** create a local `.env` file from `.env.example` and set `DATABASE_URL`. The database export is skipped when the variable is not configured. Do not commit `.env`.

In [ ]:
# Load the prepared DataFrame into PostgreSQL when DATABASE_URL is configured.
# Credentials are read from a local .env file and should never be committed.
load_dotenv(PROJECT_ROOT / '.env')
database_url = os.getenv('DATABASE_URL')

if not database_url:
    print('DATABASE_URL is not configured. Skipping PostgreSQL export.')
else:
    engine = create_engine(database_url)
    try:
        with engine.begin() as connection:
            df.to_sql(
                name='customer',
                con=connection,
                schema='public',
                if_exists='replace',
                index=False,
                chunksize=1000
            )
        print('Data successfully loaded into public.customer.')
    finally:
        engine.dispose()
